# Maxout Neuron



## 1. Introduction

Maxout (Goodfellow et al., 2013) replaces a fixed activation with the maximum over multiple learned affine functions.

For each output neuron:

\[y=\max_i(xW_i+b_i)\]

This makes the activation itself learnable.

## 2. Motivation

ReLU throws away negative values. Maxout learns the activation shape by selecting the strongest response among k linear pieces.

## 3. Mathematical Derivation

Given input \(x\in\mathbb{R}^d\), for each unit compute

\[
z_i=xW_i+b_i,\quad i=1,\ldots,k
\]

Output:

\[
y=\max(z_1,\ldots,z_k)
\]

During backpropagation only the winning affine branch receives gradient.

## 4. Advantages

- Learns piecewise-linear convex functions
- Works well with Dropout
- Avoids dead ReLUs
- High representational power

## 5. Disadvantages

- More parameters
- More memory
- More computation

In [ ]:
import torch
import torch.nn as nn

class Maxout(nn.Module):
    def __init__(self,in_features,out_features,pieces=4):
        super().__init__()
        self.out_features=out_features
        self.pieces=pieces
        self.linear=nn.Linear(in_features,out_features*pieces)

    def forward(self,x):
        y=self.linear(x)
        y=y.view(-1,self.out_features,self.pieces)
        y,_=torch.max(y,dim=2)
        return y

layer=Maxout(4,3,pieces=4)
print(layer(torch.randn(2,4)))


## Building a Classifier

In [ ]:
class MaxoutNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(
            Maxout(2,32,4),
            Maxout(32,32,4),
            nn.Linear(32,1),
            nn.Sigmoid()
        )
    def forward(self,x):
        return self.net(x)

model=MaxoutNet()
print(model)


## Train on XOR

In [ ]:
import torch.optim as optim

x=torch.tensor([[0.,0.],[0.,1.],[1.,0.],[1.,1.]])
y=torch.tensor([[0.],[1.],[1.],[0.]])

model=MaxoutNet()
loss_fn=nn.BCELoss()
opt=optim.Adam(model.parameters(),lr=0.01)

for epoch in range(2000):
    pred=model(x)
    loss=loss_fn(pred,y)
    opt.zero_grad()
    loss.backward()
    opt.step()

print(model(x).detach())


## Visualizing Maxout

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x=np.linspace(-4,4,400)

w=np.array([1.0,-0.5,2.0,-1.5])
b=np.array([0.0,1.0,-1.0,2.0])

lines=[w_i*x+b_i for w_i,b_i in zip(w,b)]
mx=np.max(lines,axis=0)

plt.figure(figsize=(8,5))
for i,l in enumerate(lines):
    plt.plot(x,l,'--',label=f'Affine {i+1}')
plt.plot(x,mx,linewidth=3,label='Maxout')
plt.legend()
plt.grid(True)
plt.title('Maxout = Maximum of Multiple Linear Functions')
plt.show()


## Comparison

| Activation | Learnable | Dead Neurons | Parameters |
|---|---:|---:|---:|
| ReLU | No | Yes | Low |
| GELU | No | No | Low |
| Maxout | Yes | No | High |

## Research Paper

Ian Goodfellow et al. (2013), *Maxout Networks* (ICML 2013). Introduced Maxout and demonstrated strong performance with dropout.

## Exercises

1. Change pieces from 2 to 8.
2. Replace Maxout with ReLU and compare.
3. Visualize decision boundaries.
4. Measure parameter count.
5. Implement Maxout without nn.Linear.

In [1]:
import torch

layer = Maxout(
    in_features=4,
    out_features=3,
    pieces=8
)

x = torch.randn(5,4)

print(layer(x))

NameError: name 'Maxout' is not defined

| Pieces | Parameters | Flexibility |  Speed |
| ------ | ---------: | ----------: | -----: |
| 2      |        Low |         Low |   Fast |
| 4      |     Medium |      Medium | Medium |
| 8      |       High |        High | Slower |
| 16     |  Very High |   Very High |   Slow |


Exe 2


In [2]:
class MaxoutNet(nn.Module):

    def __init__(self):

        super().__init__()

        self.network = nn.Sequential(

            Maxout(2,32,4),

            Maxout(32,32,4),

            nn.Linear(32,1),

            nn.Sigmoid()

        )

    def forward(self,x):

        return self.network(x)

NameError: name 'nn' is not defined

In [ ]:
class ReLUNet(nn.Module):

    def __init__(self):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(2,32),

            nn.ReLU(),

            nn.Linear(32,32),

            nn.ReLU(),

            nn.Linear(32,1),

            nn.Sigmoid()

        )

    def forward(self,x):

        return self.network(x)

In [ ]:
relu_model = ReLUNet()

maxout_model = MaxoutNet()

| Property             | ReLU     | Maxout    |
| -------------------- | -------- | --------- |
| Learnable activation | ❌       | ✅       |
| Dead neurons         | Possible | No        |
| Parameters           | Low      | High      |
| Computation          | Fast     | Slower    |
| Expressiveness       | Medium   | Very High |


## Mini Project

Train a Maxout MLP on MNIST and compare accuracy, convergence, and parameter count against ReLU and GELU models.

Inside :         --->   Projects/Maxout_MLP.ipynb